# cdm_skani CTS Demo

End-to-end demo of the generic (no-refdata) `cdm_skani` CTS tool. Skani is a fast ANI calculator
for genomes / contigs / MAGs; this image lets you compute ANI between user-supplied genomes
without bringing any reference data. For querying against the GTDB R232 representative panel,
use `cdm_skani_gtdb` instead.

- **Image:** `ghcr.io/kbaseincubator/cdm_skani:0.1.0@sha256:3c645fa64e06df1e536a48ef608656ab11c7d5d19fa690c68fee3059effe9da4`
- **Refdata:** none
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/skani_demo_20261106/`

See the [cdm_skani repo](https://github.com/kbaseincubator/cdm_skani) for full details.

**What we'll show:**
1. `skani triangle --diagonal` (all-vs-all ANI across the 4 test genomes, plus each genome against itself as a positive control)
2. `skani dist` (one query vs one reference, the building block)

The 4 test genomes are from 4 different species (2 bacteria + 2 archaea) and are far apart. skani is designed for ANI above about 80%: pairs below that are dropped by its screen (`-s`, default 80) before alignment, so unrelated genomes produce **no rows**, and lowering `--min-af` does not change that. `--diagonal` adds each genome's self-comparison (ANI 100), which proves the pipeline works end to end even when no real pair qualifies.

Re-executed 2026-11-06 after the tool audit. The earlier version claimed `--min-af 0` would produce rows; it does not, for the reason above.


## 1. Setup

In [1]:
import io, time
import pandas as pd

tscli  = get_task_service_client()
mincli = get_minio_client()

IMAGE      = "ghcr.io/kbaseincubator/cdm_skani:0.1.0@sha256:3c645fa64e06df1e536a48ef608656ab11c7d5d19fa690c68fee3059effe9da4"
OUTPUT_DIR = "cts/io/jplfaria/output/skani_demo_20261106"

print(tscli.whoami())


{'user': 'jplfaria', 'roles': ['cts_user'], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. List input genomes

Same 4 test assemblies (.fna.gz) the other tool demos use, in `cts/io/gavin/test_files/`.


In [2]:
input_files = sorted(
    f"cts/{o.object_name}"
    for o in mincli.list_objects("cts", prefix="io/gavin/test_files", recursive=True)
    if o.object_name.endswith(".fna.gz") or o.object_name.endswith(".fna")
)
print(f"{len(input_files)} input genome(s):")
for f in input_files:
    print(f"  {f}")


4 input genome(s):
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000008085.1/GCA_000008085.1_ASM808v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000010565.1/GCA_000010565.1_ASM1056v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000145985.1/GCA_000145985.1_ASM14598v1_genomic.fna.gz
  cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000147015.1/GCA_000147015.1_ASM14701v1_genomic.fna.gz


## 3. skani triangle (all-vs-all, with self-comparisons)

One container with all 4 inputs (`triangle` must see every genome at once). `-E` writes a sparse edge list (TSV) instead of a phylip matrix. `--diagonal` adds each genome against itself.

Expect exactly 4 rows: the self-comparisons at ANI 100. The 4 test genomes are below skani's ~80% ANI screen to each other, so there are no cross-genome rows. On MAG or isolate collections where many genomes are above 95% ANI, this is where the real edges show up.

In [3]:
triangle_job = tscli.submit_job(
    IMAGE,
    input_files,
    f"{OUTPUT_DIR}/triangle",
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "triangle",
        "-E",
        "--diagonal",                  # self-comparisons as a positive control
        "-o", "/out/triangle.tsv",
        "-t", "4",
        tscli.insert_files(),
    ],
    num_containers=1,
    cpus=4, memory="8GB", runtime="PT15M",
)
print(f"triangle job: {triangle_job.id}")
t0 = time.time()
triangle_job.wait_for_completion()
print(f"  state={triangle_job.get_job_status()['state']}  exit={triangle_job.get_exit_codes().get('exit_codes')}  waited={time.time()-t0:.0f}s")

tri_outs = [o for o in triangle_job.get_job().get("outputs", []) if o["file"].endswith("triangle.tsv")]
bucket, key = tri_outs[0]["file"].split("/", 1)
raw = mincli.get_object(bucket, key).read().decode("utf-8")
print(f"\ntriangle.tsv ({len(raw)} bytes):")
tri_df = pd.read_csv(io.StringIO(raw), sep="\t")
print(tri_df[["Ref_name", "Query_name", "ANI"]].to_string(index=False))


triangle job: 6212db9c-0648-42a9-b5cf-b8bf7611d98f


  state=complete  exit=[0]  waited=41s

triangle.tsv (1086 bytes):
                                                           Ref_name                                                          Query_name  ANI
AE017199.1 Nanoarchaeum equitans Kin4-M chromosome, complete genome AE017199.1 Nanoarchaeum equitans Kin4-M chromosome, complete genome  100
 AP009389.1 Pelotomaculum thermopropionicum SI DNA, complete genome  AP009389.1 Pelotomaculum thermopropionicum SI DNA, complete genome  100
        CP002098.1 Ignisphaera aggregans DSM 17230, complete genome         CP002098.1 Ignisphaera aggregans DSM 17230, complete genome  100
   CP002161.1 Candidatus Zinderia insecticola CARI, complete genome    CP002161.1 Candidatus Zinderia insecticola CARI, complete genome  100


## 4. skani dist (one query vs one reference)

The 2 bacterial genomes (Pelotomaculum and Zinderia), first file as query, second as reference. They are different phyla, far below the ~80% ANI screen, so the expected result is a header-only TSV: a correct "no qualifying pair" answer, not a failure.

In [4]:
bact_inputs = [f for f in input_files if ("000010565" in f or "000147015" in f)]
assert len(bact_inputs) == 2, f"expected 2 bacterial inputs, got {len(bact_inputs)}"
print(f"dist inputs: {bact_inputs}")

dist_job = tscli.submit_job(
    IMAGE,
    bact_inputs,
    f"{OUTPUT_DIR}/dist",
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "dist",
        "-o", "/out/dist.tsv",
        "-t", "4",
        tscli.insert_files(),
    ],
    num_containers=1,
    cpus=4, memory="8GB", runtime="PT15M",
)
print(f"dist job: {dist_job.id}")
t0 = time.time()
dist_job.wait_for_completion()
print(f"  state={dist_job.get_job_status()['state']}  exit={dist_job.get_exit_codes().get('exit_codes')}  waited={time.time()-t0:.0f}s")

dist_outs = [o for o in dist_job.get_job().get("outputs", []) if o["file"].endswith("dist.tsv")]
bucket, key = dist_outs[0]["file"].split("/", 1)
raw = mincli.get_object(bucket, key).read().decode("utf-8")
print(f"\ndist.tsv ({len(raw)} bytes):")
dist_df = pd.read_csv(io.StringIO(raw), sep="\t")
print(dist_df.to_string(index=False) if len(dist_df) else "(header only: no pair above the ANI screen, as expected for these two genomes)")


dist inputs: ['cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000010565.1/GCA_000010565.1_ASM1056v1_genomic.fna.gz', 'cts/io/gavin/test_files/collections/NONE/CDM/FastGenomics/GCA_000147015.1/GCA_000147015.1_ASM14701v1_genomic.fna.gz']


dist job: 1b330817-e699-4010-91e5-c4caba726589


  state=complete  exit=[0]  waited=521s

dist.tsv (84 bytes):
(header only: no pair above the ANI screen, as expected for these two genomes)


## 5. End-to-end check

In [5]:
checks = {
    "triangle job complete (exit 0)": triangle_job.get_job_status()["state"] == "complete"
                                          and triangle_job.get_exit_codes().get("exit_codes") == [0],
    "dist job complete (exit 0)":     dist_job.get_job_status()["state"] == "complete"
                                          and dist_job.get_exit_codes().get("exit_codes") == [0],
    "triangle has one self-row per genome at ANI 100":
        len(tri_df) == len(input_files) and (tri_df["ANI"] == 100).all()
        and (tri_df["Ref_file"] == tri_df["Query_file"]).all(),
    "dist TSV has the expected columns": {"Ref_file", "Query_file", "ANI"} <= set(dist_df.columns),
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
print("\nAll green." if all(checks.values()) else "\nSomething's off; inspect above.")


  [x] triangle job complete (exit 0)
  [x] dist job complete (exit 0)
  [x] triangle has one self-row per genome at ANI 100
  [x] dist TSV has the expected columns

All green.
